# 날씨 × 미세먼지 통합 머신러닝 분석
## 두 데이터를 결합해서 배우는 분류 + 회귀


> 예 : **"오늘 날씨를 보면 미세먼지 경보가 발령될지 예측할 수 있을까?"**

**사용 데이터**
- `weather_2020_2025.csv` : 서울(108) 일별 기상 관측 2020~2025 
- `finedust_2020_2025.csv` : 전국 미세먼지 경보 발령 기록 2020~2025 

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 분류 모델
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

# 회귀 모델
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

# 평가 지표
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

plt.rc('font', family='Malgun Gothic')
plt.rc('axes', unicode_minus=False)


### 날씨 데이터 (weather_2020_2025.csv)

기상청 종관기상관측(ASOS) 서울(stnId=108) 일별 관측값입니다.
62개 컬럼 중 사용할 핵심 컬럼

| 컬럼 | 의미 | 단위 |
|------|------|------|
| tm | 날짜 | YYYY-MM-DD |
| avgTa | **평균기온** | ℃ |
| minTa | 최저기온 | ℃ |
| maxTa | 최고기온 | ℃ |
| avgRhm | **평균습도** | % |
| avgWs | **평균풍속** | m/s |
| sumRn | 일강수량 | mm |
| avgPa | **평균현지기압** | hPa |
| sumSsHr | 합계일조시간 | hr |
| avgTca | 평균전운량(구름) | 1/10 |

In [2]:
df_w = pd.read_csv('weather_2020_2025.csv')

In [3]:
df_w = df_w[['tm', 'avgTa', 'minTa', 'maxTa', 'avgRhm','avgWs', 'sumRn', 'avgPa', 
      'sumSsHr', 'avgTca']].copy()

In [4]:
df_w.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2192 entries, 0 to 2191
Data columns (total 10 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   tm       2192 non-null   object 
 1   avgTa    2192 non-null   float64
 2   minTa    2191 non-null   float64
 3   maxTa    2192 non-null   float64
 4   avgRhm   2192 non-null   float64
 5   avgWs    2187 non-null   float64
 6   sumRn    926 non-null    float64
 7   avgPa    2191 non-null   float64
 8   sumSsHr  2185 non-null   float64
 9   avgTca   2192 non-null   float64
dtypes: float64(9), object(1)
memory usage: 171.4+ KB


In [5]:
df_w['tm'] = pd.to_datetime(df_w['tm'])

In [6]:
df_fd = pd.read_csv('finedust_2020_2025.csv')

In [7]:
df_fd.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2739 entries, 0 to 2738
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   clearVal      2739 non-null   int64 
 1   sn            2739 non-null   int64 
 2   districtName  2739 non-null   object
 3   dataDate      2739 non-null   object
 4   issueVal      2739 non-null   int64 
 5   issueTime     2739 non-null   object
 6   clearDate     2739 non-null   object
 7   issueDate     2739 non-null   object
 8   moveName      2739 non-null   object
 9   clearTime     2739 non-null   object
 10  issueGbn      2739 non-null   object
 11  itemCode      2739 non-null   object
dtypes: int64(3), object(9)
memory usage: 256.9+ KB


In [8]:
df_fd['issueDate'] = pd.to_datetime(df_fd['issueDate'])

In [9]:
df_seoul = df_fd[df_fd['districtName'] =='서울']

In [10]:
df_seoul['issueDate'].value_counts()

issueDate
2024-03-29    3
2022-12-13    3
2021-03-29    3
2021-05-07    3
2021-05-08    2
2023-04-06    2
2023-11-23    2
2023-03-23    2
2022-03-05    2
2023-04-12    2
2021-11-19    2
2025-03-25    2
2020-02-22    2
2024-01-05    1
2023-02-08    1
2025-02-16    1
2023-03-24    1
2025-04-24    1
2023-03-19    1
2023-03-11    1
2023-02-18    1
2025-11-24    1
2023-01-20    1
2024-02-01    1
2023-01-07    1
2023-01-06    1
2024-04-17    1
2024-04-16    1
2023-04-13    1
2025-12-07    1
2024-03-17    1
2024-02-11    1
2025-12-29    1
2020-12-10    1
2023-12-06    1
2023-05-21    1
2021-03-26    1
2020-10-22    1
2020-05-11    1
2020-04-22    1
2020-02-21    1
2020-02-02    1
2021-12-15    1
2021-05-24    1
2021-05-09    1
2021-04-16    1
2021-03-30    1
2021-03-15    1
2023-05-22    1
2021-03-10    1
2021-02-12    1
2021-02-06    1
2021-02-01    1
2021-01-14    1
2022-11-10    1
2022-04-27    1
2022-02-11    1
2022-01-08    1
2020-11-15    1
2025-01-20    1
Name: count, dtype: int64

In [11]:
df_seoul.groupby('issueDate')['issueVal'].agg(['max','mean','count'])

,max,mean,count
issueDate,,,
2020-02-02,76,76.000000,1
2020-02-21,78,78.000000,1
2020-02-22,157,122.000000,2
2020-04-22,199,199.000000,1
2020-05-11,181,181.000000,1
2020-10-22,152,152.000000,1
2020-11-15,79,79.000000,1
2020-12-10,79,79.000000,1
2021-01-14,174,174.000000,1


In [12]:
df_seoul.columns

Index(['clearVal', 'sn', 'districtName', 'dataDate', 'issueVal', 'issueTime',
       'clearDate', 'issueDate', 'moveName', 'clearTime', 'issueGbn',
       'itemCode'],
      dtype='object')

In [13]:
df_seoul = df_seoul.groupby('issueDate').agg(
         max_issueVal=( 'issueVal','max'),
         count_isssue= ('issueGbn', 'count')
).reset_index()

In [14]:
df_seoul

,issueDate,max_issueVal,count_isssue
0,2020-02-02,76,1
1,2020-02-21,78,1
2,2020-02-22,157,2
3,2020-04-22,199,1
4,2020-05-11,181,1
5,2020-10-22,152,1
6,2020-11-15,79,1
7,2020-12-10,79,1
8,2021-01-14,174,1
9,2021-02-01,77,1


In [15]:
def make_alert(x):
    if x>0:
        return 1
    else:
        return 0

In [16]:
df_seoul['alert'] = df_seoul['count_isssue'].apply(make_alert)

In [17]:
df_seoul

,issueDate,max_issueVal,count_isssue,alert
0,2020-02-02,76,1,1
1,2020-02-21,78,1,1
2,2020-02-22,157,2,1
3,2020-04-22,199,1,1
4,2020-05-11,181,1,1
5,2020-10-22,152,1,1
6,2020-11-15,79,1,1
7,2020-12-10,79,1,1
8,2021-01-14,174,1,1
9,2021-02-01,77,1,1


## 두 데이터 결합 (Merge)

```
weather    : 2020-01-01 | avgTa=-2.2 | avgRhm=64 | avgWs=0.6 | ...
weather    : 2020-01-02 | avgTa= 1.0 | avgRhm=65 | avgWs=1.2 | ...
             ...

finedust   : 2020-01-01 | 주의보 | PM25 | issueVal=78
finedust   : 2020-01-02 | 주의보 | PM25 | issueVal=87
finedust   : 2020-02-22 | 주의보 | PM10 | issueVal=157
             ...
```

→ **날짜(date)를 기준으로 결합**하면 "그날의 날씨 + 경보 발령 여부"를 알 수 있습니다.

### pd.merge() 와 join 종류

```python
pd.merge(left, right, on='날짜', how='left')
```

| how | 의미 | 결과 행 수 |
|-----|------|-----------|
| `left`  | 왼쪽 DataFrame의 모든 행 유지 | left 기준 |
| `inner` | 양쪽 모두 있는 날짜만 | 교집합 |
| `outer` | 양쪽 모두 포함 | 합집합 |

→ **날씨(2,192일) 기준 left join**: 경보 없는 날은 NaN → 0 처리

In [18]:
df_w.columns

Index(['tm', 'avgTa', 'minTa', 'maxTa', 'avgRhm', 'avgWs', 'sumRn', 'avgPa',
       'sumSsHr', 'avgTca'],
      dtype='object')

In [19]:
df_seoul.columns

Index(['issueDate', 'max_issueVal', 'count_isssue', 'alert'], dtype='object')

In [20]:
df = pd.merge(df_w, df_seoul,left_on='tm',right_on='issueDate', how='left' )

In [21]:
df = pd.merge(df_w, df_seoul,left_on='tm',right_on='issueDate', how='left')
df.head()

,tm,avgTa,minTa,maxTa,avgRhm,avgWs,sumRn,avgPa,sumSsHr,avgTca,issueDate,max_issueVal,count_isssue,alert
0,2020-01-01,-2.2,-6.5,0.3,64.4,0.6,0.1,1021.1,0.8,8.9,NaT,NaN,NaN,NaN
1,2020-01-02,1.0,-0.7,3.8,65.4,1.2,NaN,1018.7,0.0,7.9,NaT,NaN,NaN,NaN
2,2020-01-03,-0.1,-3.4,4.6,56.9,1.7,NaN,1016.4,8.8,0.0,NaT,NaN,NaN,NaN
3,2020-01-04,1.2,-2.8,6.1,50.8,1.9,NaN,1015.4,7.9,2.1,NaT,NaN,NaN,NaN
4,2020-01-05,1.3,-3.2,6.6,45.6,1.1,NaN,1019.7,7.1,3.9,NaT,NaN,NaN,NaN


In [22]:
df= df.drop(columns=['issueDate'])

In [25]:
df.isnull().sum()

tm              0
avgTa           0
minTa           0
maxTa           0
avgRhm          0
avgWs           0
sumRn           0
avgPa           0
sumSsHr         0
avgTca          0
max_issueVal    0
count_isssue    0
alert           0
dtype: int64

In [26]:
df= df.fillna(0)

In [27]:
df['month'] = df['tm'].dt.month

In [28]:
df.columns

Index(['tm', 'avgTa', 'minTa', 'maxTa', 'avgRhm', 'avgWs', 'sumRn', 'avgPa',
       'sumSsHr', 'avgTca', 'max_issueVal', 'count_isssue', 'alert', 'month'],
      dtype='object')

## 머신러닝 

### 분류 예제 

In [29]:
df['alert'] = df['alert'].astype(int)

In [30]:
df['alert'].value_counts()

alert
0    2132
1      60
Name: count, dtype: int64

In [31]:
X = df[['avgTa', 'minTa', 'maxTa', 'avgRhm', 'avgWs', 'sumRn', 'avgPa',
       'sumSsHr', 'avgTca', 'month']]

In [32]:
y = df['alert']

In [33]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=0,
    stratify=y
)

In [34]:
rf = RandomForestClassifier(random_state=0, class_weight='balanced')

In [35]:
rf.fit(X_train, y_train)

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [36]:
y_pred = rf.predict(X_test)

In [37]:
acc_rf = accuracy_score(y_test, y_pred)
precision_rf  = precision_score(y_test, y_pred, zero_division=0)
recall_rf  = recall_score(y_test, y_pred, zero_division=0)
f1_rf = f1_score(y_test, y_pred, zero_division=0)
print(f'정확도 (Accuracy): {acc_rf :.2f}')      
print(f'정밀도 (Precision): {precision_rf :.2f}')
print(f'재현율 (Recall): {recall_rf :.2f}')       
print(f'f1-score: {f1_rf:.2f}')  

정확도 (Accuracy): 0.97
정밀도 (Precision): 0.00
재현율 (Recall): 0.00
f1-score: 0.00


In [38]:
print( classification_report(y_test, y_pred, zero_division=0) )

              precision    recall  f1-score   support

           0       0.97      1.00      0.99       427
           1       0.00      0.00      0.00        12

    accuracy                           0.97       439
   macro avg       0.49      0.50      0.49       439
weighted avg       0.95      0.97      0.96       439



###  회귀 예제

In [39]:
df.columns

Index(['tm', 'avgTa', 'minTa', 'maxTa', 'avgRhm', 'avgWs', 'sumRn', 'avgPa',
       'sumSsHr', 'avgTca', 'max_issueVal', 'count_isssue', 'alert', 'month'],
      dtype='object')

In [40]:
df['avgRhm']

0       64.4
1       65.4
2       56.9
3       50.8
4       45.6
        ... 
2187    52.1
2188    76.4
2189    74.1
2190    45.1
2191    40.9
Name: avgRhm, Length: 2192, dtype: float64

In [41]:
X = df[['avgTa', 'minTa', 'maxTa',  'avgWs', 'sumRn', 'avgPa',
       'sumSsHr', 'avgTca','month' ]]
y = df['avgRhm']

In [42]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

In [43]:
rf = RandomForestRegressor(random_state=0)

In [44]:
rf.fit(X_train, y_train)

,n_estimators,100
,criterion,'squared_error'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [45]:
y_pred = rf.predict(X_test)

In [46]:
mae_rf = mean_absolute_error(y_test, y_pred)
rmse_rf = root_mean_squared_error(y_test, y_pred)
r2_rf = r2_score(y_test, y_pred)

print(f'평균절대오차 (MAE): {mae_rf:.2f}')
print(f'평균제곱근오차 (RMSE): {rmse_rf:.2f}')
print(f'R2 Score: {r2_rf:.2f}')

평균절대오차 (MAE): 5.85
평균제곱근오차 (RMSE): 7.71
R2 Score: 0.71


여러분 안녕하세요         
어느새 14주차 수업입니다     
이제 우리는 **프로젝트에 집중할 시간** 이 되었습니다     
그래서 오늘은 실습 내용도 적게 하여 프로젝트에 시간을 할당할 수 있도록 하였습니다        
3, 4교시는 복습 문제들을 풀어보며 편한 시간을 가지면 됩니다            
15주차도 복습 영상으로 올릴 예정입니다                  
그동안 완전 사이버 수업임에도 그 누구보다 열심히 공부해 온 여러분 **정말 사랑합니다!!**     
여러분 **자신을 믿고 시작** 해보십시요!! **잘 할 수 있습니다!!**              
저는 **언제나 여러분을 응원** 하겠습니다!! 